# 5. EDA with SQL
The SpaceX table is loaded into an in-memory SQLite database and queried with SQL (no Db2 account needed).

In [ ]:
import pandas as pd, sqlite3
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
con = sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL', con, index=False)
q = lambda sql: pd.read_sql(sql, con)
df.columns.tolist()

**Task 1 - unique launch sites**

In [ ]:
q('SELECT DISTINCT Launch_Site FROM SPACEXTBL')

**Task 2 - 5 records where launch site begins with `CCA`**

In [ ]:
q("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

**Task 3 - total payload carried by NASA (CRS) boosters**

In [ ]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'")

**Task 4 - average payload mass for booster version F9 v1.1**

In [ ]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'")

**Task 5 - date of first successful landing on a ground pad**

In [ ]:
q("SELECT MIN(Date) AS first_ground_pad_success FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'")

**Task 6 - boosters with drone-ship success and payload between 4000 and 6000 kg**

In [ ]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000")

**Task 7 - total successful and failed mission outcomes**

In [ ]:
q("SELECT Mission_Outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY Mission_Outcome")

**Task 8 - boosters that carried the maximum payload**

In [ ]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)")

**Task 9 - 2015 drone-ship failures: month, booster, launch site**
(Dates are stored as `DD-MM-YYYY` or `YYYY-MM-DD` depending on the file version; check `df['Date'].head()` and adapt the substring positions if needed.)

In [ ]:
print(df['Date'].head(3).tolist())
q("""SELECT substr(Date, 6, 2) AS month, Landing_Outcome, Booster_Version, Launch_Site
     FROM SPACEXTBL
     WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date, 0, 5) = '2015'""")

**Task 10 - rank landing outcomes between 2010-06-04 and 2017-03-20**

In [ ]:
q("""SELECT Landing_Outcome, COUNT(*) AS cnt FROM SPACEXTBL
     WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
     GROUP BY Landing_Outcome ORDER BY cnt DESC""")